![Stefanini](https://conteudo.polinize.com/content/images/2023/07/stefanini-logo.png)  
**Project     :** Stefanini Pre-Sales Support LLM
**Author      :** Ezequiel Ferreira Cardoso
**Creation    :** 2025-09-18 10:08:00  
**Description :** Notebook for 
**History     :**  
- **16/sep/2025** - EzequielFC - Creator  
- **21/sep/2025** - FlavioSS - Code review

# 1- Import

In [2]:
""" 
    Importing packages for task execution
"""

import os
from dotenv import load_dotenv


import google.generativeai as genai
from groq import Groq


from mcp.server.fastmcp import FastMCP
from mcp.types import Resource
from datetime import date, datetime, timedelta

import import_ipynb
import a_groq

# 2- Constants

In [3]:
""" 
    setting the local_context
"""

load_dotenv()

# Inicializing the gemini
genai.configure(api_key=os.getenv("GEMINI_KEY"))
gemini_client = genai  # use directy the lib as a cliente

# Inicializing Groq
groq_client = Groq(api_key=os.getenv("GROQ_KEY"))


# separar em steps, jupyter magic words
# sempre dar restart para não sujar as importações

# 3- Pre processing

In [10]:
a_groq.set_client(groq_client)

# mcp setup
mcp = FastMCP("mcp-llms")

movie_title_resource = Resource(
    uri="local://movie_title",
    name="Movie Title",
    description="Title of a movie",
    mimeType="text/plain",
    value=""
)

movie_synopsis_resource = Resource(
    uri="local://movie_synopsis",
    name="Movie Synopsis",
    description="Synopis of the movie genereted by groq",
    mimeType="text/plain",
    value=""
)

mcp.add_resource(movie_title_resource)
mcp.add_resource(movie_synopsis_resource)

# 4- Processing

In [6]:
"""
    Processing the title and genereting the synopsis
"""

@mcp.tool()
# give a title of a movie
def give_title(movie: str = "") -> str:
    title = movie
    movie_title_resource.value = title
    return title

@mcp.tool()
# genereted a synopsis of a specific movie
def generate_synopsis(style_hint: str = "", max_words: int = 60) -> str:
    if not movie_title_resource.value:
        raise RuntimeError("There is no title. Please, set at give_title()")
    
    synopsis = a_groq.generate_synopsis(
        title=movie_title_resource.value,
        max_words=max_words,
        style_hint=style_hint
    )
    movie_synopsis_resource.value = synopsis
    return synopsis

# 5- Post Processing

In [9]:
if __name__ == "__main__":
    print("=== Testing the Flow ===")
    title = give_title("Pulp Fiction")
    print("Title:", title)

    synopsis = generate_synopsis("dramatic and cinematic", max_words=40)
    print("Synopis:", synopsis)

    print("--- Resources ---")
    print("movie_title_resource:", movie_title_resource.value)
    print("movie_synopsis_resource:", movie_synopsis_resource.value)


=== Testing the Flow ===
Title: Pulp Fiction


[09/23/25 08:12:39] INFO     HTTP Request: POST https://api.groq.com/openai/v1/chat/completions     ]8;id=994327;file:///home/efcardoso/projects/labs-notebooks/mcp-context/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=149777;file:///home/efcardoso/projects/labs-notebooks/mcp-context/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Synopis: "In a world of moral decay, two hitmen, Vincent Vega and Jules Winnfield, embark on a surreal journey, intersecting with a boxer, a mob boss, and a mysterious briefcase, as fate weaves a complex web of violence, redemption, and the
--- Resources ---
movie_title_resource: Pulp Fiction
movie_synopsis_resource: "In a world of moral decay, two hitmen, Vincent Vega and Jules Winnfield, embark on a surreal journey, intersecting with a boxer, a mob boss, and a mysterious briefcase, as fate weaves a complex web of violence, redemption, and the


# 6- Execution Ending

In [8]:
"""
Ending the execution.
"""

print(f"INF: Ending Execution = {datetime.now().strftime('%d/%m/%y.%H:%M:%S')}")

INF: Ending Execution = 23/09/25.08:00:13
